In [269]:
# using Pkg
# Pkg.add(["UnPack", "LinearAlgebra", "CUDA", "SparseArrays", "Random", "QuadGK", "Printf", "StaticArrays", "PrettyTables", "CPUTime"])

# Pkg.develop([
#     PackageSpec(url="git@github.com:JanSuntajs/QHam.jl.git"),
#     PackageSpec(url="git@github.com:RockClimbingRocks/QSystem.jl.git")
# ])



In [270]:
println("Activated environment: ", Base.active_project())
include("/home/rokpintar/projects/Polfed/src/Polfed.jl")
using .Polfed

using SparseArrays, LinearAlgebra, CUDA
using QSystem

function construct_xxz_spin_sector(L::Int, delta::Real, Nup::Int)
    basis = [b for b in 0:2^L-1 if count_ones(b) == Nup] # generate basis
    dim = length(basis)
    bmap = Dict(b => i for (i, b) in enumerate(basis))  # state index map
    rows, cols, vals = Int[], Int[], Float64[]
    for (col, state) in enumerate(basis)
        for i in 1:L
            j = i % L + 1  # PBC
            si = (state >> (i - 1)) & 1
            sj = (state >> (j - 1)) & 1
            # --- S^z_i S^z_j diagonal term ---
            SzSz = (0.5 - si) * (0.5 - sj)  # spin-½: Sz = ±½
            push!(rows, col); push!(cols, col); push!(vals, delta * SzSz)
            # --- S⁺_i S⁻_j + h.c. (flip-flop term) ---
            if si != sj
                flipped = state ⊻ (1 << (i - 1)) ⊻ (1 << (j - 1))
                if haskey(bmap, flipped) 
                    push!(rows, bmap[flipped]); push!(cols, col); push!(vals, 0.5)
                end
            end
        end
    end
    return sparse(rows, cols, vals, dim, dim)
end


"""
    are_vals_in_true(vals, vals_true; atol=1e-8)

Check whether the sequence `vals` appears (within tolerance `atol`) 
as a consecutive subsequence of `vals_true`.

Returns `true` if all values match within tolerance, otherwise `false`.
"""
function are_vals_in_true(vals::AbstractVector, vals_true::AbstractVector; atol=1e-8)
    # Find the closest index in `vals_true` to vals[1]
    i_min = findmin(abs.(vals_true .- vals[1]))[2]
    i_max = i_min + length(vals) - 1

    # Ensure the slice fits inside vals_true
    if i_max > length(vals_true)
        return false
    end

    # display(abs.(vals .- view(vals_true, i_min:i_max)))

    # Check all values with broadcasting
    return all(abs.(vals .- view(vals_true, i_min:i_max)) .< atol)
end



Activated environment: /home/rokpintar/projects/Polfed/docs/Project.toml


are_vals_in_true

Lets start by constructing the matrix for the XXZ model

In [198]:
L = 14
Nup = 7
delta = 1.0
mat = construct_xxz_spin_sector(L, delta, Nup)

3432×3432 SparseMatrixCSC{Float64, Int64} with 29304 stored entries:
⎡⢻⣶⣄⠀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠲⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠙⠿⣧⡙⢦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠢⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠈⠳⣌⢿⣷⣥⡀⠀⠙⢦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠳⢤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠈⠁⠻⡿⣯⣄⠀⠀⠙⢦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⣄⠀⠀⠙⠻⣦⣀⠀⠀⠙⠦⠀⠀⠀⠀⠀⣄⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠈⠳⣄⠀⠀⠘⣻⣾⣦⡀⢀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠈⠻⢿⣷⡌⠳⣄⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠈⠹⣄⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠈⠃⠀⠐⢦⡉⠻⣦⣌⠓⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠘⠲⡀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢦⠙⣻⣾⣦⡀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠙⣆⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⢿⣷⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠈⢧⎥
⎢⢳⡀⠀⠀⠀⠀⠀⠀⠀⠙⢦⡀⠀⠀⠀⠀⠀⠀⠀⠀⢿⣷⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠹⣄⠀⠀⠀⠀⠀⠀⠀⠀⠙⢦⡀⠀⠀⠀⠀⠀⠀⠈⠻⡿⣯⣄⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠈⠦⡄⠀⠀⠀⠀⠀⠀⠀⠀⠙⢦⡀⠀⠀⠀⠀⠀⠀⢤⡙⠻⣦⣈⠳⠄⠀⢠⡀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠙⣆⡀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢦⡀⠀⠀⠀⠀⠀⠙⢦⡘⢿⣷⣦⡀⠀⠙⢦⡀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠙⢦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢦⡀⠀⠀⠀⠀⠀⠁⠈⠻⡿⣯⡄⠀⠀⠙⢦⡀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠙⢦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠀⠀⠀⠀⠀⠲⣄⠀⠀⠉⠻⣦⣄⠀⠀⠙⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠙⣻⣾⣦⢀⡀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠓⢦⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠈⢛⢿⣷⡙⢦⡀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⠢⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣌⢻⣶⣄⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⠦⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠀⠙⠿⣧⎦

Once the matrix is constructed, it is easy to call POLFED, one only needs to specify the target, number of eigenpairs, and strting iteration vector:

In [169]:
howmany = 100
target = 0.
v0 = rand(eltype(mat), size(mat, 2)); v0 /= norm(v0)  # normalize the initial vector

vals, vecs = @time Polfed.polfed(mat, v0, howmany, target)

Before resetting: left = 0.2699500062143697, right = 0.2961452492153539
After resetting: left = 0.2695331313288106, right = 0.296562124100913
  7.873005 seconds (4.14 M allocations: 300.714 MiB, 1.89% gc time, 39.87% compilation time)


([-0.058505533313523025, -0.05850553331351845, -0.05647209643218467, -0.0564720964321836, -0.0403784046520416, -0.04037840465203981, -0.03965077000443661, -0.03965077000443394, -0.03897998526494357, -0.0389799852649414  …  0.05090919997926797, 0.05090919997926908, 0.052853576418356724, 0.05285357641835948, 0.056791753207639795, 0.056791753207644194, 0.057324688066133396, 0.05732468806614081, 0.06063664230415606, 0.06063664230416056], [0.0011951548387337879 0.001749425781631378 … -2.9942419449713107e-5 -0.002613971234050933; -0.0030578102680667474 -0.004475915542425322 … 7.304044084638674e-5 0.006376425646414809; … ; -0.003057810268066716 -0.004475915542425618 … -7.304044084604177e-5 -0.006376425646414517; 0.0011951548387337412 0.0017494257816314984 … 2.994241944960171e-5 0.0026139712340508122])

Lets scheck if the eigenvalues match the ones from exact diagonalization:

In [170]:
vals_true = eigvals(Matrix(mat));
println(are_vals_in_true(vals, vals_true; atol=1e-10))
println(are_vals_in_true(vals, vals_true; atol=1e-12))
println(are_vals_in_true(vals, vals_true; atol=1e-14))

false
false
false


Eigenvalues match up to $10^{-14}$.  
To see what is happenenig one can also see the information report of the run, by simply setting one of the optional keyword argument to true

In [171]:
vals, vecs, report = Polfed.polfed(mat, v0, howmany, target; produce_report=true)
Polfed.display_report(report)

Before resetting: left = 0.26992185792087453, right = 0.2961733975088491
After resetting: left = 0.2695331313288106, right = 0.296562124100913
Spectral Transformation Report:
- Targeted  100  eigenpairs at energy 0.283048
- Exposing ev's in the interval [0.269533, 0.296562], with width δ = 0.02703
- Performing 'Chebyshev' spectral transformation of order  K = 189  (and order safety factor 0.97)
- Matrix multiplication performed  41_869  times! With parallelization strategy:  MulColsParallel() 
- Optimization with Clenshaw recurrence is  disabled !
Factorization Report: (LanczosFactorization with blocksize 1)
- Number of converged eigenpairs:    100  (out of  100  requested)
- Lanczos convergence satisfied by:  100  (with tolerance  1.00e-14  max residual was  1.77e-16 )
- Eigen convergence satisfied by:    100  (with tolerance  1.00e-09  max residual was  3.00e-13 )
- Iterations needed:  221   (out of  238  reserved, overestimated by  7.14% )
- Basis type: MatrixBasis and reorthogonali

Should i dsplay rescaled or non rescaled versoion of the interval and delta?

Here you can see lots of details about the polfed run like, order of polynomial expansioin, number of matrix-vector multiplications, howmany eigenpairs have converged, largest residual, number of iterations needed, timings and more. You can also get more detailed report about the convergence by calling

In [172]:
Polfed.display_report(report; show_convergence_details=true)

Spectral Transformation Report:
- Targeted  100  eigenpairs at energy 0.283048
- Exposing ev's in the interval [0.269533, 0.296562], with width δ = 0.02703
- Performing 'Chebyshev' spectral transformation of order  K = 189  (and order safety factor 0.97)
- Matrix multiplication performed  41_869  times! With parallelization strategy:  MulColsParallel() 
- Optimization with Clenshaw recurrence is  disabled !
Factorization Report: (LanczosFactorization with blocksize 1)
- Number of converged eigenpairs:    100  (out of  100  requested)
- Lanczos convergence satisfied by:  100  (with tolerance  1.00e-14  max residual was  1.77e-16 )
- Eigen convergence satisfied by:    100  (with tolerance  1.00e-09  max residual was  3.00e-13 )
- Iterations needed:  221   (out of  238  reserved, overestimated by  7.14% )
- Basis type: MatrixBasis and reorthogonalization technique: FullRO()
- Convergence check was peerformed 9 times, here is the table of results:
╭──────────┬─────────────┬───────────┬────

Altghou timing of XXX seconds for getting the eignpairs for XXZ model of system size $L=18$ is very good compered to exact diagonalization, we can do much better, instead of using only one thread one can use several of them! Running julia with say $4$ threads can give us a $4$ times speedup. This can be simply done by setting the block size of the initial block to equal number of threads. We need to be carefull to ensure that all vectors in the starting block are orthogonal to one another, below this is done by using the QR decompositon.

In [173]:
v0_ = rand(size(mat,1), Threads.nthreads())
v0 = Matrix(qr(v0_).Q)
vals, vecs, report = Polfed.polfed(mat, v0, howmany, target; produce_report=true)
Polfed.display_report(report)

Before resetting: left = 0.27006315073131487, right = 0.2960321046984093
After resetting: left = 0.2696832456266777, right = 0.29641200980304644
Spectral Transformation Report:
- Targeted  100  eigenpairs at energy 0.283048
- Exposing ev's in the interval [0.269683, 0.296412], with width δ = 0.02673
- Performing 'Chebyshev' spectral transformation of order  K = 191  (and order safety factor 0.97)
- Matrix multiplication performed  43_266  times! With parallelization strategy:  MulColsParallel() 
- Optimization with Clenshaw recurrence is  disabled !
Factorization Report: (BlockLanczosFactorization with blocksize 1)
- Number of converged eigenpairs:    100  (out of  100  requested)
- Lanczos convergence satisfied by:  100  (with tolerance  1.00e-14  max residual was  3.07e-21 )
- Eigen convergence satisfied by:    100  (with tolerance  1.00e-09  max residual was  2.36e-13 )
- Iterations needed:  226   (out of  238  reserved, overestimated by  5.04% )
- Basis type: MatrixBasis and reorth

# Optimization of the mapping

This is connected to the next important subsection of the paralization \ref{sec:Paralization_of_the_mapping}. The timing of YYY seconds is now much better! But we can still do even better! Paralization is only one of the possibilitie for improvements, anther one is to design a costum mapping, for a specific model. One can take advantage of having all of the offdiagonal elements at value $0.5$. It is ussually beneficial to seperate diagonal and offdiagonalpard, because mapping of thte diagonal elements of the matrix is simple and fast because of the coallesacal memmory acces.

In [174]:
function mapvec_xxz_onthefly!(
    Y::AbstractVector,
    X::AbstractVector,
    basis::Vector{Int},
    state_to_index::Dict{Int,Int},
    diags::Vector{Float64},
    L::Int,
    offdiag_val::Real,
)
@inbounds for (i, state) in enumerate(basis)
    acc = 0
    for site in 1:L
        jsite = site % L + 1
        si = (state >> (site - 1)) & 1
        sj = (state >> (jsite - 1)) & 1
        if si != sj
            flipped = state ⊻ (1 << (site - 1)) ⊻ (1 << (jsite - 1))
            j = state_to_index[flipped]
            acc += X[j]
        end
    end    
    Y[i] = diags[i] * X[i] + offdiag_val*acc
end
end

mapvec_xxz_onthefly! (generic function with 1 method)

Exploiting the fact that POLFED like Lanczos is again only depandend on the mapping! Therfore mapping can replace the hamiltonian, and polfed function exploits that fact! Therfore curently we have two main entrypoints:

In [175]:
Polfed.polfed(mat::AbstractMatrix, x0::AbstractVecOrMat, howmany::Int, target::Real)
Polfed.polfed(f!::Function, x0::AbstractVecOrMat, howmany::Int, target::Real)

UndefVarError: UndefVarError: `x0` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

We are sticking to julia's convention about denoting inplace function with the exlemation point at the end of the name, \juliainline{f!} in our case above. The input function shouhld have two input parameters, first the resulting vector and second the vector one needs to map (following the julia's mul!() convention). 

In [176]:
basis = [b for b in 0:2^L-1 if count_ones(b) == Nup]
state_to_index = Dict(s => i for (i, s) in enumerate(basis))
diags = [mat[i,i] for i in eachindex(basis)]
offdiag_val = 0.5*0.5^delta
xxz_vec_mapping!(Y,X) = mapvec_xxz_onthefly!(Y, X, basis, state_to_index, diags, L, offdiag_val)
v0 = rand(size(mat,1)); v0 ./= norm(v0)

vals, vecs, report = Polfed.polfed(xxz_vec_mapping!, v0, howmany, target; produce_report=true)
Polfed.display_report(report)

Before resetting: left = 0.22916993966946164, right = 0.2601260440672354
After resetting: left = 0.22873096636185208, right = 0.26056501737484494
Spectral Transformation Report:
- Targeted  100  eigenpairs at energy 0.244648
- Exposing ev's in the interval [0.228731, 0.260565], with width δ = 0.03183
- Performing 'Chebyshev' spectral transformation of order  K = 163  (and order safety factor 0.97)
- Matrix multiplication performed  38_568  times! With parallelization strategy:  MulColsParallel() 
- Optimization with Clenshaw recurrence is  disabled !
Factorization Report: (LanczosFactorization with blocksize 1)
- Number of converged eigenpairs:    100  (out of  100  requested)
- Lanczos convergence satisfied by:  100  (with tolerance  1.00e-14  max residual was  1.39e-15 )
- Eigen convergence satisfied by:    100  (with tolerance  1.00e-09  max residual was  1.37e-12 )
- Iterations needed:  236   (out of  238  reserved, overestimated by  0.84% )
- Basis type: MatrixBasis and reorthogon

# Partial reorthogonalization (PRO)

In [247]:
using CUDA, CUDA.CUSPARSE

L = 16
Nup = L÷2
delta = 1.0
cumat = CUDA.CUSPARSE.CuSparseMatrixCSR(construct_xxz_spin_sector(L, delta, Nup))
howmany = 1500

v0_ = rand(size(cumat,1))
# v0 = CuMatrix(Matrix(qr(v0_).Q))
v0 = CuVector(v0_ ./ norm(v0_))
vals, vecs, report = Polfed.polfed(cumat, v0, howmany, target; produce_report=true)
Polfed.display_report(report)

Before resetting: left = 0.23388473292671974, right = 0.3301455636867219
After resetting: left = 0.23316561778866585, right = 0.33086467882477577
Spectral Transformation Report:
- Targeted  1500  eigenpairs at energy 0.282015
- Exposing ev's in the interval [0.233166, 0.330865], with width δ = 0.09770
- Performing 'Chebyshev' spectral transformation of order  K = 52  (and order safety factor 0.97)
- Matrix multiplication performed  135_452  times! With parallelization strategy:  NoParallel() 
- Optimization with Clenshaw recurrence is  disabled !
Factorization Report: (LanczosFactorization with blocksize 1)
- Number of converged eigenpairs:    1500  (out of  1500  requested)
- Lanczos convergence satisfied by:  1500  (with tolerance  1.00e-14  max residual was  1.78e-15 )
- Eigen convergence satisfied by:    1500  (with tolerance  1.00e-09  max residual was  3.66e-11 )
- Iterations needed:  2576   (out of  3206  reserved, overestimated by  19.65% )
- Basis type: MatrixBasis and reortho

In [256]:

lc= Polfed.LanczosConfig(;rot=Polfed.PartialRO(4))
stc = Polfed.SpectralTransformConfig(;overestimate_iters=1.3)
v0_ = rand(size(cumat,1))
v0 = CuVector(v0_ ./ norm(v0_))

vals, vecs, report = Polfed.polfed(cumat, v0, howmany, target; 
    produce_report=true,
    lanczos = lc,
    spectral_transform = stc,
    )
Polfed.display_report(report)

Before resetting: left = 0.2339169939407048, right = 0.3301133026727376
After resetting: left = 0.23316561778866624, right = 0.33086467882477616
Spectral Transformation Report:
- Targeted  1500  eigenpairs at energy 0.282015
- Exposing ev's in the interval [0.233166, 0.330865], with width δ = 0.09770
- Performing 'Chebyshev' spectral transformation of order  K = 52  (and order safety factor 0.97)
- Matrix multiplication performed  135_452  times! With parallelization strategy:  NoParallel() 
- Optimization with Clenshaw recurrence is  disabled !
Factorization Report: (LanczosFactorization with blocksize 1)
- Number of converged eigenpairs:    1500  (out of  1500  requested)
- Lanczos convergence satisfied by:  1500  (with tolerance  1.00e-14  max residual was  8.13e-16 )
- Eigen convergence satisfied by:    1500  (with tolerance  1.00e-09  max residual was  7.99e-11 )
- Iterations needed:  2576   (out of  3334  reserved, overestimated by  22.74% )
- Basis type: MatrixBasis and reorthog

In [263]:
v0_ = rand(size(cumat,1), 1)
v0 = CuMatrix(Matrix(qr(v0_).Q))

vals, vecs, report = Polfed.polfed(cumat, v0, howmany, target; 
    produce_report=true,
    lanczos = lc,
    spectral_transform = stc,
    )
Polfed.display_report(report)

Before resetting: left = 0.23400732477986236, right = 0.3300229718335809
After resetting: left = 0.23333108028256316, right = 0.3306992163308801
Spectral Transformation Report:
- Targeted  1500  eigenpairs at energy 0.282015
- Exposing ev's in the interval [0.233331, 0.330699], with width δ = 0.09737
- Performing 'Chebyshev' spectral transformation of order  K = 53  (and order safety factor 0.97)
- Matrix multiplication performed  136_703  times! With parallelization strategy:  NoParallel() 
- Optimization with Clenshaw recurrence is  disabled !
Factorization Report: (BlockLanczosFactorization with blocksize 1)
- Number of converged eigenpairs:    1500  (out of  1500  requested)
- Lanczos convergence satisfied by:  1500  (with tolerance  1.00e-14  max residual was  1.00e-15 )
- Eigen convergence satisfied by:    1500  (with tolerance  1.00e-09  max residual was  6.72e-11 )
- Iterations needed:  2551   (out of  3334  reserved, overestimated by  23.49% )
- Basis type: MatrixBasis and reo

In [264]:
v0_ = rand(size(cumat,1), 2)
v0 = CuMatrix(Matrix(qr(v0_).Q))

vals, vecs, report = Polfed.polfed(cumat, v0, howmany, target; 
    produce_report=true,
    lanczos = lc,
    spectral_transform = stc,
    )
Polfed.display_report(report)

Before resetting: left = 0.233979936445587, right = 0.33005036016785416
After resetting: left = 0.2333310802825621, right = 0.33069921633087906
Spectral Transformation Report:
- Targeted  1500  eigenpairs at energy 0.282015
- Exposing ev's in the interval [0.233331, 0.330699], with width δ = 0.09737
- Performing 'Chebyshev' spectral transformation of order  K = 53  (and order safety factor 0.97)
- Matrix multiplication performed  137_286  times! With parallelization strategy:  NoParallel() 
- Optimization with Clenshaw recurrence is  disabled !
Factorization Report: (BlockLanczosFactorization with blocksize 2)
- Number of converged eigenpairs:    87  (out of  1500  requested)
- Lanczos convergence satisfied by:  1500  (with tolerance  1.00e-14  max residual was  4.35e-16 )
- Eigen convergence satisfied by:    87  (with tolerance  1.00e-09  max residual was  1.25e-06 )
- Iterations needed:  1281   (out of  1681  reserved, overestimated by  23.80% )
- Basis type: MatrixBasis and reorthog

┌ Info: Eigenvalues converged: 87 out of 1500
└ @ Main.Polfed.Lanczos /home/rokpintar/projects/Polfed/src/Lanczos/Convergence/CalculatingResiduals.jl:30


In [265]:
v0_ = rand(size(cumat,1), 3)
v0 = CuMatrix(Matrix(qr(v0_).Q))

vals, vecs, report = Polfed.polfed(cumat, v0, howmany, target; 
    produce_report=true,
    lanczos = lc,
    spectral_transform = stc,
    )
Polfed.display_report(report)

Before resetting: left = 0.23400392731510983, right = 0.3300263692983328
After resetting: left = 0.23333108028256283, right = 0.3306992163308798
Spectral Transformation Report:
- Targeted  1500  eigenpairs at energy 0.282015
- Exposing ev's in the interval [0.233331, 0.330699], with width δ = 0.09737
- Performing 'Chebyshev' spectral transformation of order  K = 53  (and order safety factor 0.97)
- Matrix multiplication performed  139_194  times! With parallelization strategy:  NoParallel() 
- Optimization with Clenshaw recurrence is  disabled !
Factorization Report: (BlockLanczosFactorization with blocksize 3)
- Number of converged eigenpairs:    97  (out of  1500  requested)
- Lanczos convergence satisfied by:  1500  (with tolerance  1.00e-14  max residual was  9.54e-15 )
- Eigen convergence satisfied by:    97  (with tolerance  1.00e-09  max residual was  5.37e-07 )
- Iterations needed:  866   (out of  1129  reserved, overestimated by  23.29% )
- Basis type: MatrixBasis and reorthog

┌ Info: Eigenvalues converged: 97 out of 1500
└ @ Main.Polfed.Lanczos /home/rokpintar/projects/Polfed/src/Lanczos/Convergence/CalculatingResiduals.jl:30


In [266]:
v0_ = rand(size(cumat,1), 4)
v0 = CuMatrix(Matrix(qr(v0_).Q))

vals, vecs, report = Polfed.polfed(cumat, v0, howmany, target; 
    produce_report=true,
    lanczos = lc,
    spectral_transform = stc,
    )
Polfed.display_report(report)

Before resetting: left = 0.234014522040717, right = 0.3300157745727223
After resetting: left = 0.23333108028256117, right = 0.3306992163308781
Spectral Transformation Report:


┌ Info: Eigenvalues converged: 496 out of 1500
└ @ Main.Polfed.Lanczos /home/rokpintar/projects/Polfed/src/Lanczos/Convergence/CalculatingResiduals.jl:30


- Targeted  1500  eigenpairs at energy 0.282015
- Exposing ev's in the interval [0.233331, 0.330699], with width δ = 0.09737
- Performing 'Chebyshev' spectral transformation of order  K = 53  (and order safety factor 0.97)
- Matrix multiplication performed  141_632  times! With parallelization strategy:  NoParallel() 
- Optimization with Clenshaw recurrence is  disabled !
Factorization Report: (BlockLanczosFactorization with blocksize 4)
- Number of converged eigenpairs:    496  (out of  1500  requested)
- Lanczos convergence satisfied by:  1500  (with tolerance  1.00e-14  max residual was  2.48e-15 )
- Eigen convergence satisfied by:    496  (with tolerance  1.00e-09  max residual was  3.68e-09 )
- Iterations needed:  661   (out of  854  reserved, overestimated by  22.60% )
- Basis type: MatrixBasis and reorthogonalization technique: PartialRO(4)
Timings: Percentages are distributed as: (Mapping, Reorthogonalization, Convergence check, others)
- Total polfed run took:  31.44 seconds  

# Testing HybridMatrixBasis

In [275]:
using CUDA, CUDA.CUSPARSE

L = 20
Nup = L÷2
delta = 1.0
cumat = CUDA.CUSPARSE.CuSparseMatrixCSR(construct_xxz_spin_sector(L, delta, Nup))
howmany = 250

v0_ = rand(size(cumat,1))
v0 = CuVector(v0_ ./ norm(v0_))

lc= Polfed.LanczosConfig(;rot=Polfed.PartialRO(4), basistype=Polfed.HybridMatrixBasis)
stc = Polfed.SpectralTransformConfig(;overestimate_iters=1.3)

vals, vecs, report = Polfed.polfed(cumat, v0, howmany, target; 
    produce_report=true,
    lanczos = lc,
    spectral_transform = stc,
    )
Polfed.display_report(report)

Before resetting: left = 0.2803096239943562, right = 0.2812953805108128
After resetting: left = 0.28029424599600417, right = 0.28131075850916487
Effective GPU memory usage: 93.66% (36.897 GiB/39.394 GiB)
Memory pool usage: 935.227 MiB (23.594 GiB reserved)
Reserved memmory of OB type HybridMatrixBasis: 
   Free GPU memmory: 2.680619008 GB
   Reserved GPU memmory: 0.854311744 GB
   Number of vectors reserved: 578
   Free CPU memmory: 1038.941384704 GB
   Number of vectors reserved: 0
   Reserved CPU memmory: 0.0 GB
Spectral Transformation Report:
- Targeted  250  eigenpairs at energy 0.280803
- Exposing ev's in the interval [0.280294, 0.281311], with width δ = 0.00102
- Performing 'Chebyshev' spectral transformation of order  K = 5043  (and order safety factor 0.97)
- Matrix multiplication performed  2_915_104  times! With parallelization strategy:  NoParallel() 
- Optimization with Clenshaw recurrence is  disabled !
Factorization Report: (LanczosFactorization with blocksize 1)
- Number

In [274]:
using CUDA, CUDA.CUSPARSE

L = 20
Nup = L÷2
delta = 1.0
cumat = CUDA.CUSPARSE.CuSparseMatrixCSR(construct_xxz_spin_sector(L, delta, Nup))
howmany = 750

v0_ = rand(size(cumat,1))
v0 = CuVector(v0_ ./ norm(v0_))

lc= Polfed.LanczosConfig(;rot=Polfed.PartialRO(4), basistype=Polfed.HybridMatrixBasis)
stc = Polfed.SpectralTransformConfig(;overestimate_iters=1.3)

vals, vecs, report = Polfed.polfed(cumat, v0, howmany, target; 
    produce_report=true,
    lanczos = lc,
    spectral_transform = stc,
    )
Polfed.display_report(report)

Before resetting: left = 0.2793261920590465, right = 0.28227881244612385
After resetting: left = 0.27928150347662617, right = 0.2823235010285442
Effective GPU memory usage: 93.66% (36.897 GiB/39.394 GiB)
Memory pool usage: 9.606 GiB (23.594 GiB reserved)
Reserved memmory of OB type HybridMatrixBasis: 
   Free GPU memmory: 2.680619008 GB
   Reserved GPU memmory: 2.412174336 GB
   Number of vectors reserved: 1632
   Free CPU memmory: 1035.717758976 GB
   Number of vectors reserved: 49
   Reserved CPU memmory: 0.072424352 GB
Starting to fill CPU basis storage!
Spectral Transformation Report:
- Targeted  750  eigenpairs at energy 0.280803
- Exposing ev's in the interval [0.279282, 0.282324], with width δ = 0.00304
- Performing 'Chebyshev' spectral transformation of order  K = 1685  (and order safety factor 0.97)
- Matrix multiplication performed  2_833_235  times! With parallelization strategy:  NoParallel() 
- Optimization with Clenshaw recurrence is  disabled !
Factorization Report: (Lan